# Percolation Benchmark Analysis

This notebook is a thin interactive wrapper around `percolation_uart.analysis`.

It keeps the SQLite loading and plotting logic in the analysis module, so the notebook can stay lightweight while you inspect sessions, select a run, and render the standard plots inline.

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

from IPython.display import Image, display

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'python' / 'percolation_uart' / 'analysis.py').exists():
            return candidate
    raise FileNotFoundError('could not locate repository root')

repo_root = find_repo_root()
python_dir = repo_root / 'python'
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

from percolation_uart import analysis

db_path = analysis.DEFAULT_DB
plot_dir = repo_root / 'python' / 'output' / 'notebook_analysis'
plot_dir.mkdir(parents=True, exist_ok=True)
db_path, plot_dir

In [ ]:
conn = analysis._connect(db_path)
try:
    sessions = analysis.list_sessions(conn)
    print(analysis.summarize_db(conn))
    print()
    for index, session in enumerate(sessions, start=1):
        payload = session.payload
        print(f'{index:2d}. session_id={session.session_id} created_at={session.created_at}')
        if 'config_hash' in payload:
            print(f'    config_hash={payload["config_hash"]}')
        if 'args' in payload and isinstance(payload['args'], dict):
            args = payload['args']
            print(f'    runs={args.get("runs")} repeats={args.get("repeats")} points={args.get("points")} steps={args.get("steps")}')
finally:
    conn.close()

In [ ]:
selected_session_index = -1

conn = analysis._connect(db_path)
try:
    sessions = analysis.list_sessions(conn)
    if not sessions:
        raise RuntimeError(f'no sessions found in {db_path}')
    selected_session = sessions[selected_session_index]
    selected_session_id = selected_session.session_id
    summary_rows = analysis.load_summary_rows(conn, session_id=selected_session_id)
    raw_rows = analysis.load_raw_rows(conn, session_id=selected_session_id)
finally:
    conn.close()

print(f'selected_session_id={selected_session_id}')
print(f'selected_created_at={selected_session.created_at}')
print(f'summary_rows={len(summary_rows)} raw_rows={len(raw_rows)}')

In [ ]:
def render_png(path: Path) -> None:
    if not path.exists():
        print(f'missing: {path}')
        return
    display(Image(filename=str(path)))

session_plot_dir = plot_dir / selected_session_id
session_plot_dir.mkdir(parents=True, exist_ok=True)

analysis.plot_dashboard(summary_rows, raw_rows, session_plot_dir / 'dashboard.png')
analysis.plot_front_density(raw_rows, session_plot_dir / 'front_density.png')
analysis.plot_cluster_mass(raw_rows, session_plot_dir / 'cluster_mass.png')
analysis.plot_occupancy_bias(raw_rows, session_plot_dir / 'occupancy_bias.png')
analysis.plot_core_latency(raw_rows, session_plot_dir / 'core_latency.png')
analysis.plot_spanning_probability(raw_rows, session_plot_dir / 'spanning_probability.png')

for filename in [
    'dashboard.png',
    'front_density.png',
    'cluster_mass.png',
    'occupancy_bias.png',
    'core_latency.png',
    'spanning_probability.png',
]:
    print(filename)
    render_png(session_plot_dir / filename)

## Optional batch view

Change `selected_session_index` and rerun the selection and rendering cells to inspect another benchmark session.

If you want to compare several sessions at once, keep the analysis logic in the module and use a small loop here to render one subdirectory per session.